# Dual-Basis Segmented-Arbitrage Carry Portfolio (DB-SACP)

A robust research notebook for the full DB-SACP plumbing:
**data loading → validation/alignment → spreads → signals → EWMA risk model → constrained sizing → diagnostics**.

This notebook is built to be fail-fast and explicit on schemas and units.


## Data Checklist (required for real mode)

**Units:** all rates/yields/spreads in **decimals** (`0.05 = 5%`).

| Dataset | Expected file names (`data/`) | Required columns | Freq | Used for |
|---|---|---|---|---|
| UST nominal yields | `ust_yields_daily.parquet` or `.csv` | `date`, `y_ust_2y`, `y_ust_5y`, `y_ust_10y`, `y_ust_20y` | daily business | nominal leg + spread base |
| OIS fixed rates | `ois_rates_daily.parquet` or `.csv` | `date`, `r_ois_2y`, `r_ois_5y`, `r_ois_10y`, `r_ois_20y` | daily business | Treasury–OIS spread |
| TIPS real yields | `tips_real_yields_daily.parquet` or `.csv` | `date`, `y_tips_real_2y`, `y_tips_real_5y`, `y_tips_real_10y`, `y_tips_real_20y` | daily business | TIPS basis real leg |
| ZC inflation swaps | `infl_zc_swap_daily.parquet` or `.csv` | `date`, `pi_zc_2y`, `pi_zc_5y`, `pi_zc_10y`, `pi_zc_20y` | daily business | synthetic nominal leg |
| DV01 map (recommended) | `dv01_map.parquet` or `.csv` | `tenor_y`, `dv01_ust_per_1mm`, `dv01_tips_per_1mm` | static | sizing realism |

**No silent fill policy:** this notebook aligns using **date intersection** and does not forward fill by default.


## Quickstart

- **Demo mode (no external files):** set `USE_DEMO_DATA=True` (default) and run all cells.
- **Real mode:** set `USE_DEMO_DATA=False`, place files in `data/` with exact names/schemas above.
- If real files are missing or malformed, notebook raises clear errors with exact expected paths/columns.


In [ ]:
from __future__ import annotations
import math
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
np.random.seed(7)


In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TENORS_Y = [2, 5, 10, 20]
USE_DEMO_DATA = True
AUTO_CONVERT_PERCENT_TO_DECIMAL = True

Z_ENTRY = 1.25
Z_EXIT = 0.50
EWMA_LAMBDA = 0.97
CONE_Q = 0.90
EWMA_COV_LAMBDA = 0.985
RIDGE = 1e-8

COST_SWAP_ANNUAL = {2: 0.0005, 5: 0.0005, 10: 0.0007, 20: 0.0010}
COST_TIPS_ANNUAL = {2: 0.0010, 5: 0.0012, 10: 0.0015, 20: 0.0020}

EQUITY_USD = 10_000_000.0
LEVERAGE_CAP_GROSS = 4.0
HAIRCUT_UST = 0.02
HAIRCUT_TIPS = 0.04
IM_RATE_OIS = 0.015
IM_RATE_INFL = 0.020
LIQUIDITY_BUFFER_PCT = 0.30
W_CAP = 0.35


In [ ]:
def tenor_cols(prefix: str, tenors: List[int]) -> List[str]:
    return [f"{prefix}_{m}y" for m in tenors]

def read_table_any(base: Path) -> pd.DataFrame:
    cands = [base.with_suffix('.parquet'), base.with_suffix('.csv')]
    for p in cands:
        if p.exists():
            if p.suffix == '.parquet':
                return pd.read_parquet(p)
            return pd.read_csv(p)
    expected = " or ".join(str(p.resolve()) for p in cands)
    raise FileNotFoundError(f"Missing required input file. Expected one of: {expected}")

def ensure_schema(df: pd.DataFrame, required: Iterable[str], name: str) -> pd.DataFrame:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns={missing}; present={list(df.columns)}")
    return df

def coerce_date_index(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    if out['date'].isna().any():
        raise ValueError(f"{name}: found {int(out['date'].isna().sum())} unparseable dates")
    out = out.sort_values('date')
    if out['date'].duplicated().any():
        raise ValueError(f"{name}: duplicate dates detected ({int(out['date'].duplicated().sum())})")
    out = out.set_index('date')
    if not out.index.is_monotonic_increasing:
        raise ValueError(f"{name}: date index must be monotonic increasing")
    return out

def coerce_numeric(df: pd.DataFrame, cols: List[str], name: str) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    if out[cols].isna().all(axis=None):
        raise ValueError(f"{name}: all required numeric columns are NaN after coercion")
    return out

def validate_and_fix_units(df: pd.DataFrame, cols: List[str], name: str, auto_convert: bool=True):
    out = df.copy()
    vals = out[cols].stack().dropna()
    q99 = float(vals.abs().quantile(0.99))
    if q99 > 1.0:
        if auto_convert:
            out[cols] = out[cols] / 100.0
            return out, f"{name}: detected percent-like levels (|q99|={q99:.3f}); auto-converted /100"
        raise ValueError(f"{name}: values look percent-like (|q99|={q99:.3f}). Provide decimals.")
    if float(vals.min()) < -0.05 or float(vals.max()) > 0.20:
        raise ValueError(f"{name}: unit sanity failed with range [{float(vals.min()):.4f}, {float(vals.max()):.4f}] expected decimal-like")
    return out, f"{name}: units validated as decimals"

def make_demo_data(tenors: List[int]) -> Dict[str, pd.DataFrame]:
    idx = pd.date_range('2015-01-01', '2025-01-01', freq='B')
    n = len(idx)
    lvl = np.cumsum(np.random.normal(0, 0.0003, n)) + 0.025
    slp = np.cumsum(np.random.normal(0, 0.0002, n))
    ust, ois, tips, infl = (pd.DataFrame(index=idx) for _ in range(4))
    for m in tenors:
        b = (m - 2) / 18
        core = lvl + 0.7 * b * slp + np.random.normal(0, 0.0002, n)
        ust[f'y_ust_{m}y'] = core
        ois[f'r_ois_{m}y'] = core - (0.0015 + 0.0004*np.random.randn(n))
        tips[f'y_tips_real_{m}y'] = core - (0.020 + 0.002*np.random.randn(n))
        infl[f'pi_zc_{m}y'] = 0.020 + 0.001*np.random.randn(n)
    dv01 = pd.DataFrame({'tenor_y': tenors, 'dv01_ust_per_1mm': [190, 450, 850, 1400], 'dv01_tips_per_1mm': [180, 420, 800, 1300]})
    return {'ust': ust, 'ois': ois, 'tips': tips, 'infl': infl, 'dv01': dv01}

def load_inputs(data_dir: Path, tenors: List[int], use_demo: bool=True, auto_convert: bool=True):
    notes = []
    if use_demo:
        return make_demo_data(tenors), ["DEMO MODE: synthetic data generated"]
    specs = {'ust': ('ust_yields_daily', tenor_cols('y_ust', tenors)), 'ois': ('ois_rates_daily', tenor_cols('r_ois', tenors)), 'tips': ('tips_real_yields_daily', tenor_cols('y_tips_real', tenors)), 'infl': ('infl_zc_swap_daily', tenor_cols('pi_zc', tenors))}
    out = {}
    for k, (base, cols) in specs.items():
        raw = read_table_any(data_dir / base)
        raw = ensure_schema(raw, ['date'] + cols, k.upper())
        raw = coerce_numeric(raw, cols, k.upper())
        clean = coerce_date_index(raw[['date'] + cols], k.upper())
        clean, msg = validate_and_fix_units(clean, cols, k.upper(), auto_convert=auto_convert)
        notes.append(msg)
        out[k] = clean
    try:
        dv01 = read_table_any(data_dir / 'dv01_map')
        dv01 = ensure_schema(dv01, ['tenor_y', 'dv01_ust_per_1mm', 'dv01_tips_per_1mm'], 'DV01')
        out['dv01'] = dv01.copy()
    except FileNotFoundError:
        out['dv01'] = None
        notes.append('DV01 map not found: will use fallback duration proxy')
    return out, notes


In [ ]:
def alignment_diagnostics(dfs: Dict[str, pd.DataFrame], tenors: List[int]):
    rows = []
    for k in ['ust', 'ois', 'tips', 'infl']:
        df = dfs[k]
        rows.append({'dataset': k, 'rows': len(df), 'first_date': df.index.min(), 'last_date': df.index.max(), 'is_monotonic': bool(df.index.is_monotonic_increasing), 'duplicate_dates': int(df.index.duplicated().sum())})
    summary = pd.DataFrame(rows).set_index('dataset')
    idx_intersection = dfs['ust'].index
    idx_union = dfs['ust'].index
    for k in ['ois', 'tips', 'infl']:
        idx_intersection = idx_intersection.intersection(dfs[k].index)
        idx_union = idx_union.union(dfs[k].index)
    miss_rows = []
    for k, pref in [('ust','y_ust'), ('ois','r_ois'), ('tips','y_tips_real'), ('infl','pi_zc')]:
        cols = tenor_cols(pref, tenors)
        tmp = dfs[k][cols].isna().mean().rename('missing_pct').reset_index().rename(columns={'index':'series'})
        tmp['dataset'] = k
        miss_rows.append(tmp)
    missingness = pd.concat(miss_rows, ignore_index=True)
    return summary, idx_intersection, idx_union, missingness

def build_spreads(ust, ois, tips, infl, tenors):
    s_swap, s_tips = pd.DataFrame(index=ust.index), pd.DataFrame(index=ust.index)
    for m in tenors:
        s_swap[m] = ois[f'r_ois_{m}y'] - ust[f'y_ust_{m}y']
        s_tips[m] = (tips[f'y_tips_real_{m}y'] + infl[f'pi_zc_{m}y']) - ust[f'y_ust_{m}y']
    return s_swap, s_tips

def ewma_vol(x: pd.Series, lam: float) -> pd.Series:
    dx = x.diff()
    var = 0.0
    out = []
    for i, v in enumerate(dx.values):
        if i == 0 or np.isnan(v):
            out.append(np.nan); continue
        var = lam * var + (1.0 - lam) * float(v*v)
        out.append(math.sqrt(max(var, 0.0)))
    return pd.Series(out, index=x.index)

def rolling_z(x: pd.Series, win: int = 252) -> pd.Series:
    m = x.rolling(win, min_periods=max(20, win//3)).mean()
    s = x.rolling(win, min_periods=max(20, win//3)).std(ddof=0)
    return (x - m) / s.replace(0, np.nan)

def ewma_cov_latest(dS: pd.DataFrame, lam: float=0.985, ridge: float=1e-8) -> np.ndarray:
    X = dS.dropna().values
    if len(X) < 2: raise ValueError('Not enough observations for covariance')
    n = X.shape[1]
    cov = np.zeros((n, n))
    for row in X:
        v = row.reshape(-1, 1)
        cov = lam * cov + (1 - lam) * (v @ v.T)
    return cov + ridge * np.eye(n)

def max_notional_from_capital(equity, buffer_pct, h_ust, h_tips, im_ois, im_infl, weights: pd.Series):
    avail = equity * (1.0 - buffer_pct)
    usage = 0.0
    drivers = []
    for (k, tenor), w in weights.items():
        if w <= 0: continue
        rate = (h_ust + im_ois) if k == 'swap' else (h_tips + h_ust + im_infl)
        usage += w * rate
        drivers.append((k, tenor, w, rate, w*rate))
    tbl = pd.DataFrame(drivers, columns=['strategy','tenor','weight','resource_rate','weighted_usage'])
    return (0.0 if usage <= 0 else avail / usage), usage, tbl


In [ ]:
def run_pipeline(use_demo: bool=True):
    data, notes = load_inputs(DATA_DIR, TENORS_Y, use_demo=use_demo, auto_convert=AUTO_CONVERT_PERCENT_TO_DECIMAL)
    diag_summary, idx_inter, idx_union, miss_tbl = alignment_diagnostics(data, TENORS_Y)
    if len(idx_inter) == 0: raise ValueError('No overlapping dates across required datasets after intersection alignment.')
    ust, ois, tips, infl = data['ust'].loc[idx_inter], data['ois'].loc[idx_inter], data['tips'].loc[idx_inter], data['infl'].loc[idx_inter]
    S_swap, S_tips = build_spreads(ust, ois, tips, infl, TENORS_Y)
    mu_swap = pd.DataFrame({m: np.maximum(0.0, -S_swap[m]) - COST_SWAP_ANNUAL[m] for m in TENORS_Y}, index=idx_inter)
    mu_tips = pd.DataFrame({m: np.abs(S_tips[m]) - COST_TIPS_ANNUAL[m] for m in TENORS_Y}, index=idx_inter)
    vol_swap = pd.DataFrame({m: ewma_vol(S_swap[m], EWMA_LAMBDA) for m in TENORS_Y}, index=idx_inter)
    vol_tips = pd.DataFrame({m: ewma_vol(S_tips[m], EWMA_LAMBDA) for m in TENORS_Y}, index=idx_inter)
    z_swap = pd.DataFrame({m: rolling_z(S_swap[m]) for m in TENORS_Y}, index=idx_inter)
    z_tips = pd.DataFrame({m: rolling_z(S_tips[m]) for m in TENORS_Y}, index=idx_inter)
    riskoff_swap = vol_swap > vol_swap.rolling(252, min_periods=84).quantile(CONE_Q)
    riskoff_tips = vol_tips > vol_tips.rolling(252, min_periods=84).quantile(CONE_Q)

    def gen_sig(z, mu, riskoff, z_entry, z_exit):
        sig = pd.DataFrame(0.0, index=z.index, columns=z.columns)
        for m in z.columns:
            enter = (mu[m] > 0) & (z[m].abs() >= z_entry) & (~riskoff[m].fillna(False))
            exit_ = (z[m].abs() <= z_exit) | (riskoff[m].fillna(False))
            active = False; out = []
            for t in z.index:
                if (not active) and bool(enter.loc[t]): active = True
                elif active and bool(exit_.loc[t]): active = False
                out.append(1.0 if active else 0.0)
            sig[m] = out
        return sig

    sig_swap = gen_sig(z_swap, mu_swap, riskoff_swap, Z_ENTRY, Z_EXIT)
    sig_tips = pd.DataFrame(0.0, index=idx_inter, columns=TENORS_Y)
    for m in TENORS_Y:
        enter = (mu_tips[m] > 0) & (z_tips[m].abs() >= Z_ENTRY) & (~riskoff_tips[m].fillna(False))
        exit_ = (z_tips[m].abs() <= Z_EXIT) | (riskoff_tips[m].fillna(False))
        active = False; side = 0.0; out = []
        for t in idx_inter:
            if (not active) and bool(enter.loc[t]):
                active = True; side = 1.0 if float(np.sign(S_tips.loc[t, m])) >= 0 else -1.0
            elif active and bool(exit_.loc[t]):
                active = False; side = 0.0
            out.append(side if active else 0.0)
        sig_tips[m] = out

    dS = pd.concat([S_swap.add_prefix('swap_'), S_tips.add_prefix('tips_')], axis=1).diff()
    mu_vec = pd.concat([mu_swap.add_prefix('swap_').iloc[-1], mu_tips.add_prefix('tips_').iloc[-1]])
    Sigma = ewma_cov_latest(dS, lam=EWMA_COV_LAMBDA, ridge=RIDGE)
    w_raw = np.linalg.solve(Sigma, mu_vec.values)
    w = np.clip(w_raw, 0.0, None)
    if w.sum() > 0: w = w / w.sum()
    w = np.minimum(w, W_CAP)
    if w.sum() > 0: w = w / w.sum()
    labels = [('swap', int(c.split('_')[1])) for c in mu_swap.add_prefix('swap_').columns] + [('tips', int(c.split('_')[1])) for c in mu_tips.add_prefix('tips_').columns]
    w_series = pd.Series(w, index=pd.MultiIndex.from_tuples(labels, names=['strategy','tenor_y']), name='weight')
    max_notional, usage, usage_tbl = max_notional_from_capital(EQUITY_USD, LIQUIDITY_BUFFER_PCT, HAIRCUT_UST, HAIRCUT_TIPS, IM_RATE_OIS, IM_RATE_INFL, w_series)
    max_notional = min(max_notional, LEVERAGE_CAP_GROSS * EQUITY_USD)
    return {'notes': notes, 'diag_summary': diag_summary, 'missingness': miss_tbl, 'idx_inter': idx_inter, 'idx_union': idx_union, 'S_swap': S_swap, 'S_tips': S_tips, 'mu_swap': mu_swap, 'mu_tips': mu_tips, 'vol_swap': vol_swap, 'vol_tips': vol_tips, 'z_swap': z_swap, 'z_tips': z_tips, 'riskoff_swap': riskoff_swap, 'riskoff_tips': riskoff_tips, 'sig_swap': sig_swap, 'sig_tips': sig_tips, 'w': w_series, 'usage': usage, 'usage_tbl': usage_tbl, 'max_notional': max_notional, 'data': data}

results = run_pipeline(USE_DEMO_DATA)
print('Pipeline complete. N aligned dates:', len(results['idx_inter']))


In [ ]:
print('--- Validation Notes ---')
for n in results['notes']:
    print('-', n)
print('\n--- Dataset Alignment Diagnostics ---')
display(results['diag_summary'])
print('\nIntersection N=', len(results['idx_inter']), '| Union N=', len(results['idx_union']))
print('\n--- Missingness by dataset/series ---')
display(results['missingness'])
print('\n--- Resource usage drivers ---')
display(results['usage_tbl'])
print(f"Weighted resource usage={results['usage']:.4%}, max_notional={results['max_notional']:,.0f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
for m in TENORS_Y:
    ax[0].plot(results['S_swap'].index, 10_000 * results['S_swap'][m], label=f'{m}y')
    ax[1].plot(results['S_tips'].index, 10_000 * results['S_tips'][m], label=f'{m}y')
ax[0].set_title('Treasury–OIS spread (bp)')
ax[1].set_title('TIPS synthetic basis (bp)')
for a in ax:
    a.grid(True); a.legend(ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
# 1) schema presence
assert set(results['S_swap'].columns) == set(TENORS_Y)
assert set(results['S_tips'].columns) == set(TENORS_Y)

# 2) alignment intersection behavior
raw_data, _ = load_inputs(DATA_DIR, TENORS_Y, use_demo=True)
_, idx_i, idx_u, _ = alignment_diagnostics(raw_data, TENORS_Y)
assert len(idx_i) <= len(idx_u)
assert results['idx_inter'].equals(idx_i)

# 3) spread correctness
t0 = results['idx_inter'][10]; m0 = TENORS_Y[0]
assert np.isclose(results['S_swap'].loc[t0, m0], raw_data['ois'].loc[t0, f'r_ois_{m0}y'] - raw_data['ust'].loc[t0, f'y_ust_{m0}y'])

# 4) EWMA vol checks
vol_test = ewma_vol(pd.Series([1.0, 1.1, np.nan, 1.05]), 0.97)
assert (vol_test.dropna() >= 0).all()
assert vol_test.isna().sum() >= 1

# 5) weight constraints
w = results['w']
assert (w >= -1e-12).all()
assert (w <= W_CAP + 1e-10).all()
if w.sum() > 0: assert np.isclose(w.sum(), 1.0)

print('All notebook tests passed.')


In [ ]:
try:
    _ = run_pipeline(use_demo=False)
    print('Real mode loaded data successfully.')
except FileNotFoundError as e:
    print('Expected fail-fast in real mode (missing files):')
    print(str(e))


## Final Notes

### What changed
- Reworked notebook into a fully runnable end-to-end pipeline with explicit schema/unit requirements, strict date alignment by intersection, and validation diagnostics.
- Added robust parquet/csv loading, fail-fast schema/date/unit checks, and optional percent-to-decimal auto-conversion.
- Replaced slow rolling covariance extraction with an EWMA latest-covariance estimator plus ridge stabilization.
- Added constrained sizing, explicit resource usage decomposition, and targeted in-notebook assert tests.

### Placeholder assumptions
- Cost drags, haircuts/margins, and DV01 fallback are placeholders and should be calibrated.
- Signal thresholds and EWMA parameters are demonstration defaults.

### Data needed for production realism
- Production-grade cleaned rates series and calendars.
- Instrument-level DV01, financing schedules, and transaction-cost assumptions.
